In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Bệnh trầm cảm có liên quan đến số lượng điếu thuốc lá được hút ở những người trẻ tuổi không?

In [ ]:
import numpy
import pandas
import statsmodels.formula.api as smf
import statsmodels.stats.multicomp as multi

In [ ]:
data = pandas.read_csv('/kaggle/input/nesarc-huit/nesarc.csv', low_memory=False)

In [ ]:
data['S3AQ3B1'] = pandas.to_numeric(data['S3AQ3B1'], errors='coerce')
data['S3AQ3C1'] = pandas.to_numeric(data['S3AQ3C1'], errors='coerce')
data['CHECK321'] = pandas.to_numeric(data['CHECK321'], errors='coerce')

In [ ]:
# Người trẻ tuổi có hút thuốc
sub1=data[(data['AGE']>=18) & (data['AGE']<=25) & (data['CHECK321']==1)]

In [ ]:
sub1['S3AQ3B1']=sub1['S3AQ3B1'].replace(9, numpy.nan)
sub1['S3AQ3C1']=sub1['S3AQ3C1'].replace(99, numpy.nan)

recode1 = {1: 30, 2: 22, 3: 14, 4: 5, 5: 2.5, 6: 1}
sub1['USFREQMO']= sub1['S3AQ3B1'].map(recode1)
sub1['USFREQMO'] = pandas.to_numeric(sub1['USFREQMO'], errors='coerce')

sub1['NUMCIGMO_EST']=sub1['USFREQMO'] * sub1['S3AQ3C1']
sub1['NUMCIGMO_EST']= pandas.to_numeric(sub1['NUMCIGMO_EST'], errors='coerce')

In [ ]:
# NUMCIGMO_EST: ước lượng số lượng thuốc lá hút mỗi tháng
# MAJORDEPLIFE: trầm cảm hay không
ct1 = sub1.groupby('NUMCIGMO_EST').size()
print (ct1)

In [ ]:
# Phân tích ANOVA
model1 = smf.ols(formula='NUMCIGMO_EST ~ C(MAJORDEPLIFE)', data=sub1)
results1 = model1.fit()

In [ ]:
print(results1.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sub1['Depression_Status'] = sub1['MAJORDEPLIFE'].map({0: 'Không trầm cảm', 1: 'Trầm cảm'})

# Vẽ biểu đồ boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(x='Depression_Status', y='NUMCIGMO_EST', data=sub1, palette='Set2')

# Tùy chỉnh biểu đồ
plt.xlabel('Trạng thái trầm cảm')
plt.ylabel('Ước lượng số điếu thuốc hút mỗi tháng')
plt.show()

# ========= Biến giải thích nhiều hơn 2 loại

In [ ]:
sub3 = sub1[['NUMCIGMO_EST', 'ETHRACE2A']].dropna()
print(sub3['ETHRACE2A'].value_counts().sort_index())

In [ ]:
model2 = smf.ols(formula='NUMCIGMO_EST ~ C(ETHRACE2A)', data=sub3).fit()

In [ ]:
print(model2.summary())

In [ ]:
# Vẽ biểu đồ boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(x='ETHRACE2A', y='NUMCIGMO_EST', data=sub3, palette='Set2')

plt.xlabel('Sắc tộc')
plt.ylabel('Ước lượng số điếu thuốc hút mỗi tháng')
plt.show()

# Phân tích sâu

In [ ]:
mc1 = multi.MultiComparison(sub3['NUMCIGMO_EST'], sub3['ETHRACE2A'])

In [ ]:
res1 = mc1.tukeyhsd()

In [ ]:
print(res1.summary())